# Evaluación Sumativa 2 — Validación por remuestreo y simulación
## Dataset: *E-Commerce Shipping Data* (entregas atrasadas)

**MCDI501 - Estadística Computacional para la Toma de Decisiones**
Magíster en Ciencia de Datos e Inteligencia Artificial, Universidad Andrés Bello
Equipo: José Ignacio Rodríguez, Yerko Gallardo Miranda y Sebastián Rojas Araya

Este cuaderno valida **todos** los resultados de la Sumativa 1 (S1) de cada familia mediante bootstrap, permutación, simulación Monte Carlo, jackknife y análisis de sensibilidad, y prepara insumos verificables para la Sumativa 3 (S3). Sigue el orden de la planificación del equipo (versión 5):

0. Configuración, datos y semilla
1. Punto de partida, alcance y cobertura
2. Bootstrap y pruebas de permutación
3. Correlaciones, asociaciones y simulación Monte Carlo
4. Robustez, sensibilidad e insumos para S3
5. Comprobaciones de aceptación

El cuaderno desarrolla la cobertura completa. El informe presenta en su cuerpo principal (máximo siete páginas) los hallazgos más relevantes, y en los anexos A–F el desarrollo completo.

### Trabajo en equipo sobre este cuaderno

Todo el análisis se desarrolla en este único cuaderno, sin módulos externos. Para que los tres integrantes puedan trabajar en paralelo sin conflictos en Git:

1. **Cada integrante edita solo las celdas de su sección** y, en la sección «Funciones compartidas», solo las funciones de las que es responsable.
2. **Durante el desarrollo se commitea sin salidas** («Clear All Outputs» en VS Code o Jupyter): las salidas cambian en cada ejecución y son la principal fuente de conflictos.
3. **Se trabaja con PR pequeños y frecuentes**, cada uno con revisión cruzada.
4. **Revisión final de todo el equipo:** al cerrar, los tres revisan el cuaderno completo; luego se ejecuta desde un kernel reiniciado, con todas las celdas en orden, y esa ejecución, con sus salidas, es la que se entrega.

## 0. Configuración, datos y semilla

Los datos se leen directamente desde S1, sin copiarlos: `sumativa_1/data/raw/ecommerce_shipping.csv`. Antes de usarlos se verifica su huella SHA-256 contra `sumativa_1/docs/registro_fuente.json`, que acepta la forma original (CRLF) y la normalizada (LF) del archivo.

La base de trabajo `df` usa los mismos nombres de variables y la misma codificación que S1: `entrega_atrasada = 1` para atraso y `0` para entrega puntual; las diferencias entre grupos se calculan como **atrasadas menos puntuales**. No se eliminan filas ni se modifica el CSV.

In [ ]:
from pathlib import Path
import hashlib
import json

import pandas as pd

# Raíz de la Sumativa 2: funciona al ejecutar desde el repositorio, sumativa_2/ o sus notebooks/
ubicacion = Path.cwd().resolve()
candidatas = [p for a in [ubicacion, *ubicacion.parents] for p in [a, a / 'sumativa_2']]
RAIZ = next((p for p in candidatas if (p / 'notebooks/sumativa_2_grupo1.ipynb').is_file()
             and (p / 'docs/registro_semilla.json').is_file()), None)
if RAIZ is None:
    raise FileNotFoundError('Ejecute desde el repositorio, sumativa_2/ o sus notebooks/.')
RAIZ_S1 = RAIZ.parent / 'sumativa_1'
DATOS = RAIZ_S1 / 'data/raw/ecommerce_shipping.csv'

# Integridad: el CSV debe ser el registrado en S1 (forma original CRLF o normalizada LF)
registro_fuente = json.loads((RAIZ_S1 / 'docs/registro_fuente.json').read_text(encoding='utf-8'))
HUELLA_DATOS = hashlib.sha256(DATOS.read_bytes()).hexdigest()
assert HUELLA_DATOS in {registro_fuente['file']['sha256'], registro_fuente['file']['sha256_lf_normalized']}, \
    'El CSV no coincide con el registrado en S1.'

raw = pd.read_csv(DATOS, encoding='utf-8-sig')

NOMBRES = {
    'ID': 'id_envio', 'Warehouse_block': 'bloque_bodega', 'Mode_of_Shipment': 'modalidad_envio',
    'Customer_care_calls': 'llamadas_atencion', 'Customer_rating': 'calificacion_cliente',
    'Cost_of_the_Product': 'costo_producto', 'Prior_purchases': 'compras_previas',
    'Product_importance': 'importancia_producto', 'Gender': 'genero',
    'Discount_offered': 'descuento_ofrecido', 'Weight_in_gms': 'peso_gramos',
}
TARGET = 'entrega_atrasada'

df = raw.rename(columns=NOMBRES)
df[TARGET] = raw['Reached.on.Time_Y.N']
df['importancia_producto'] = pd.Categorical(df['importancia_producto'],
                                            categories=['low', 'medium', 'high'], ordered=True)

n = len(df)
n_atr, n_punt = int(df[TARGET].sum()), int((df[TARGET] == 0).sum())
assert (n, n_atr, n_punt) == (10_999, 6_563, 4_436), 'Los conteos no coinciden con S1.'

print(f'Datos: {DATOS.relative_to(RAIZ.parent).as_posix()} (SHA-256 {HUELLA_DATOS[:12]}…)')
print(f'Dimensiones: {n:,} filas x {df.shape[1]} columnas | atrasadas: {n_atr:,} | puntuales: {n_punt:,}'.replace(',', '.'))
df.head()

### Semilla

Toda la aleatoriedad del cuaderno usa la **semilla única** registrada según `docs/protocolo_semilla.md`, obtenida del *beacon* de aleatoriedad de la Universidad de Chile con una regla publicada antes de la emisión del pulso. Se lee desde `docs/registro_semilla.json`: no se escribe a mano, no se reemplaza por otra (por ejemplo, 42) ni se genera una nueva.

Aquí solo se comprueba que la semilla corresponde al `outputValue` del pulso registrado. La verificación completa contra el *beacon* se ejecuta con `uv run python sumativa_2/scripts/semilla_beacon.py verificar`.

In [ ]:
registro_semilla = json.loads((RAIZ / 'docs/registro_semilla.json').read_text(encoding='utf-8'))
SEMILLA = registro_semilla['semilla']

# La semilla debe derivarse del pulso registrado: int(outputValue, 16) mod 2**32
assert SEMILLA == int(registro_semilla['pulso']['outputValue'], 16) % 2**32, \
    'La semilla no corresponde al pulso registrado.'

print(f'SEMILLA = {SEMILLA}')
print(f"Pulso: {registro_semilla['pulso']['uri']} ({registro_semilla['pulso']['timeStamp']})")

### Flujos aleatorios

Cada módulo recibe su propio generador, derivado de la semilla con `np.random.SeedSequence(SEMILLA).spawn(k)`. Así, los resultados de un módulo no dependen de las llamadas aleatorias de otro, y cada sección puede ejecutarse o revisarse por separado.

El orden de los flujos es fijo: no se reordena, y un flujo nuevo se agrega siempre al final. Cada sección obtiene su generador con `generador('<nombre>')`, que entrega los mismos números cada vez que la sección se ejecuta.

| Flujo | Nombre | Módulo | Sección |
|---|---|---|---|
| 0 | `bootstrap_parametros` | Bootstrap de todos los parámetros y efectos | 2.1 |
| 1 | `permutaciones` | Permutaciones de las siete pruebas | 2.2 |
| 2 | `bootstrap_coeficientes` | Bootstrap de las 20 correlaciones y 15 asociaciones | 3.1 |
| 3 | `montecarlo_lotes` | Monte Carlo de lotes de envíos | 3.2 |
| 4 | `montecarlo_empirico` | Monte Carlo empírico de todos los estimandos (Anexo D) | 3.2 |
| 5 | `sensibilidad_escenarios` | Bootstrap y permutaciones de los escenarios de sensibilidad | 4.2 |
| 6 | `sensibilidad_montecarlo` | Sensibilidad de Monte Carlo | 4.3 |

El *jackknife* (4.1) es determinista y no usa flujo. Junto con el mapa de flujos se registran la semilla, n, B, N, α, las versiones de los paquetes, los tiempos y las advertencias.

In [ ]:
import numpy as np

# Orden fijo de los flujos: no se reordena; un flujo nuevo se agrega siempre al final
FLUJOS = [
    'bootstrap_parametros',      # 2.1  Yerko
    'permutaciones',             # 2.2  Sebastián
    'bootstrap_coeficientes',    # 3.1  Sebastián
    'montecarlo_lotes',          # 3.2  José
    'montecarlo_empirico',       # 3.2  José (Anexo D)
    'sensibilidad_escenarios',   # 4.2  Yerko y Sebastián
    'sensibilidad_montecarlo',   # 4.3  José
]
SEMILLAS_FLUJO = dict(zip(FLUJOS, np.random.SeedSequence(SEMILLA).spawn(len(FLUJOS))))


def generador(flujo):
    """Generador nuevo del flujo indicado; al reejecutar una sección se obtienen los mismos números."""
    return np.random.default_rng(SEMILLAS_FLUJO[flujo])


pd.DataFrame({'flujo': range(len(FLUJOS)), 'nombre': FLUJOS}).set_index('flujo')

### Funciones compartidas

Varias secciones reutilizan los mismos procedimientos; por ejemplo, la sensibilidad (4.2) repite el *bootstrap*, las permutaciones y las correlaciones en cada escenario. Esas funciones se definen **aquí, una sola vez y antes de su primer uso**. Propuesta inicial, a confirmar por el equipo:

| Función prevista | Uso | Secciones que la usan | Responsable |
|---|---|---|---|
| *Bootstrap* de un estimando | Réplicas, IC percentil y BCa, procesadas por lotes | 2.1, 3.2 (Anexo D), 4.2 | Yerko |
| *Jackknife* | Aceleración del BCa, error estándar e influencia de cada envío | 2.1, 4.1 | Yerko |
| Permutación de un contraste | Estadístico recalculado, distribución nula y $p_{perm}$ | 2.2, 4.2 | Sebastián |
| *Bootstrap* de coeficientes | Correlaciones y V de Cramér con filas emparejadas | 3.1, 4.2 | Sebastián |
| Simulación de lotes | Lotes binomiales, cuantiles, probabilidades y MCSE | 3.2, 4.3 | José |
| Exportación | Tablas y figuras en `outputs/` con nombres y columnas fijos | Todas | José |

**Reglas:**

- Antes de desarrollar, se acuerdan los nombres, los parámetros (datos, generador, B) y las columnas de salida de cada función.
- Cada función recibe su generador como parámetro; ninguna crea generadores propios ni usa otra semilla.
- Si una firma debe cambiar, se hace en un PR corto y aparte, avisando a quienes la usan.

*Pendiente: confirmar responsables y firmas antes de iniciar los módulos.*

In [ ]:
# --- Funciones compartidas de remuestreo (Yerko): bootstrap, jackknife e intervalos ---
from scipy.stats import norm


def bootstrap(estadistico, muestras, rng, B=10_000, lote=1_000):
    """Réplicas bootstrap de `estadistico`, procesadas por lotes.

    estadistico: función que recibe una remuestra por cada muestra, cada una con forma (lote, n_g),
                 y devuelve (lote,) o (lote, k). Con k > 1, los k estadísticos comparten remuestras.
    muestras:    tupla de arreglos 1D. Una muestra = filas completas; varias = dentro de cada grupo.
    rng:         generador del flujo de la sección, por ejemplo generador('bootstrap_parametros').
    """
    muestras = [np.asarray(m) for m in muestras]
    replicas = []
    for inicio in range(0, B, lote):
        b = min(lote, B - inicio)
        # Cada grupo se remuestrea con reemplazo y con su propio tamaño
        remuestras = [m[rng.integers(0, len(m), size=(b, len(m)))] for m in muestras]
        replicas.append(np.asarray(estadistico(*remuestras)))
    return np.concatenate(replicas)


def jackknife(estadistico, muestras, lote=1_000):
    """Valores leave-one-out: `estadistico` recalculado al retirar cada observación, grupo por grupo.

    Usa la misma función que bootstrap(). Al retirar una observación de un grupo, los demás quedan
    completos. Devuelve (n_total,) o (n_total, k), en el orden de las muestras y sus observaciones.
    """
    muestras = [np.asarray(m) for m in muestras]
    valores = []
    for g, m in enumerate(muestras):
        n_g = len(m)
        posiciones = np.arange(n_g - 1)
        for inicio in range(0, n_g, lote):
            retiradas = np.arange(inicio, min(inicio + lote, n_g))
            # Fila r: todas las posiciones del grupo excepto la retirada
            indices = posiciones[None, :] + (posiciones[None, :] >= retiradas[:, None])
            entradas = [np.broadcast_to(o, (len(retiradas), len(o))) for o in muestras]
            entradas[g] = m[indices]
            valores.append(np.asarray(estadistico(*entradas)))
    return np.concatenate(valores)


def intervalos_bootstrap(replicas, estimacion, valores_jk, tamanos=None, alfa=0.05):
    """Sesgo, EE, z0, aceleración, IC percentil y BCa de un estimando.

    tamanos: tamaño de cada grupo, en el orden usado por jackknife() (None = una sola muestra).
    """
    rep = np.asarray(replicas, dtype=float)
    B = len(rep)
    res = {'B': B, 'sesgo': rep.mean() - estimacion, 'ee': rep.std(ddof=1),
           'ic_percentil_inf': np.quantile(rep, alfa / 2),
           'ic_percentil_sup': np.quantile(rep, 1 - alfa / 2),
           'z0': np.nan, 'aceleracion': np.nan, 'ic_bca_inf': np.nan, 'ic_bca_sup': np.nan,
           'advertencia': ''}

    # z0 con empates: (#{réplicas < estimación} + 0,5 #{réplicas = estimación}) / B
    prop = (np.sum(rep < estimacion) + 0.5 * np.sum(rep == estimacion)) / B

    # Aceleración por jackknife; con varios grupos, cada uno se pondera por su tamaño
    tamanos = [len(valores_jk)] if tamanos is None else list(tamanos)
    num = den = 0.0
    for v in np.split(np.asarray(valores_jk, dtype=float), np.cumsum(tamanos)[:-1]):
        u = (len(v) - 1) * (v.mean() - v)
        num += np.sum(u ** 3) / len(v) ** 3
        den += np.sum(u ** 2) / len(v) ** 2

    if rep.min() == rep.max():
        res['advertencia'] = 'Réplicas constantes: distribución degenerada; BCa no estimable'
    elif den == 0:
        res['advertencia'] = 'Jackknife sin variación: aceleración y BCa no estimables'
    elif prop in (0, 1):
        res['advertencia'] = 'Estimación fuera del rango de las réplicas: BCa no estimable'
    else:
        z0, a = norm.ppf(prop), num / (6 * den ** 1.5)
        z = norm.ppf([alfa / 2, 1 - alfa / 2])
        niveles = norm.cdf(z0 + (z0 + z) / (1 - a * (z0 + z)))
        res.update(z0=z0, aceleracion=a, ic_bca_inf=np.quantile(rep, niveles[0]),
                   ic_bca_sup=np.quantile(rep, niveles[1]))
    return res

### Convención de figuras

Se usa la misma convención de la Sumativa 1: paleta, tamaños al ancho del informe (`ANCHO` = 6,3 pulgadas y `ANCHO_MEDIO` = 4,2), tipografía serif, etiquetas en español y números con formato es-CL. Cada figura se exporta con `guardar(nombre)` en PDF (para LaTeX) y PNG, en `outputs/figuras/`.

**Nombres:** `fig_<técnica>_<n>_<tema>`, con la técnica de cada sección: `boot` (2.1), `perm` (2.2), `cor` (3.1), `mc` (3.2 y 4.3), `jk` (4.1) y `sens` (4.2). Por ejemplo, `fig_boot_1_parametros` o `fig_perm_1_t02`.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Convención de figuras de S1
AZUL  = '#1f5f8b'   # entregas a tiempo / serie principal
TERRA = '#d1603d'   # entregas atrasadas
GRIS  = '#555555'   # referencias: tasa global, medias, umbrales
BG    = '#ffffff'   # fondo de los ejes
GRID  = '#e1e4e8'   # líneas de grilla

ANCHO, ANCHO_MEDIO = 6.3, 4.2   # pulgadas: ancho de texto del informe y 2/3 de él

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'pdf.fonttype': 42,
    'figure.constrained_layout.use': True,
    'font.family': 'serif', 'font.size': 9,
    'axes.titlesize': 9, 'axes.labelsize': 8.5, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'legend.fontsize': 8, 'legend.frameon': False,
    'axes.facecolor': BG, 'axes.edgecolor': '#8c8578',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titleweight': 'normal', 'axes.titlelocation': 'left',
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.axisbelow': True,
})
(RAIZ / 'outputs/figuras').mkdir(parents=True, exist_ok=True)

ETIQ_VAR = {
    'llamadas_atencion': 'Llamadas a atención', 'costo_producto': 'Costo (USD)',
    'compras_previas': 'Compras previas', 'descuento_ofrecido': 'Descuento ofrecido',
    'peso_gramos': 'Peso (g)', 'bloque_bodega': 'Bloque de bodega',
    'modalidad_envio': 'Modalidad de envío', 'calificacion_cliente': 'Calificación del cliente',
    'importancia_producto': 'Importancia del producto', 'genero': 'Género', TARGET: 'Entrega atrasada',
}
ETIQ_CORTA = {
    'llamadas_atencion': 'Llamadas', 'costo_producto': 'Costo', 'compras_previas': 'Compras previas',
    'descuento_ofrecido': 'Descuento', 'peso_gramos': 'Peso', 'bloque_bodega': 'Bodega',
    'modalidad_envio': 'Modalidad', 'calificacion_cliente': 'Calificación',
    'importancia_producto': 'Importancia', 'genero': 'Género',
}
ETIQ_CAT = {
    'modalidad_envio': {'Flight': 'Avión', 'Road': 'Carretera', 'Ship': 'Barco'},
    'importancia_producto': {'low': 'Baja', 'medium': 'Media', 'high': 'Alta'},
    'genero': {'F': 'Femenino', 'M': 'Masculino'},
}


def etiq(v):
    return ETIQ_VAR.get(v, v)


def etiq_cat(v, valores):
    m = ETIQ_CAT.get(v, {})
    return [m.get(str(x), str(x)) for x in valores]


def num(x, dec=0):
    '''Número con formato es-CL: punto de miles y coma decimal.'''
    return f'{x:,.{dec}f}'.replace(',', '_').replace('.', ',').replace('_', '.')


FMT_MILES = FuncFormatter(lambda x, _: num(x))
FMT_PCT = FuncFormatter(lambda x, _: f'{num(x)}%')


def rejilla(k, ncols=3, alto=1.9, ancho_total=ANCHO, **kw):
    '''Figura con k paneles en ncols columnas al ancho del informe; devuelve también los ejes sobrantes.'''
    nfilas = int(np.ceil(k / ncols))
    fig, ax = plt.subplots(nfilas, ncols, figsize=(ancho_total, alto * nfilas), squeeze=False, **kw)
    return fig, ax.flat[:k], ax.flat[k:]


def guardar(nombre, fig=None, formatos=('pdf', 'png')):
    '''Exporta la figura en PDF (vectorial, para LaTeX) y PNG (vista rápida).'''
    fig = fig or plt.gcf()
    for ext in formatos:
        fig.savefig(RAIZ / f'outputs/figuras/{nombre}.{ext}')
    plt.show()

## 1. Punto de partida, alcance y cobertura

**Objetivo.** Validar resultados propios de S1 sobre atrasos en envíos de comercio electrónico, cuantificar su incertidumbre y su sensibilidad, y preparar insumos verificables para S3. Los valores de referencia son resultados de S1, no de S2.

**Referencias de S1.** Se leen desde `sumativa_1/outputs/tablas/` y se registran en `docs/manifiesto_s1.json`, con el commit, las rutas y las huellas de cada archivo. Los cuatro parámetros inferenciales principales son la proporción de atraso (IC de Wilson) y las medias de peso, costo y llamadas (IC *t* de Student), en `estimaciones.csv`.

### Regla de cobertura completa

Cada técnica se aplica a **todos los resultados de S1 de la familia a la que corresponde**. Ningún resultado se excluye del cálculo porque no se destaque en el cuerpo del informe. El inventario comprende:

- las cuatro estimaciones inferenciales principales;
- todos los IC y efectos de los contrastes;
- las estadísticas descriptivas y las proporciones por grupo;
- las siete pruebas;
- 20 coeficientes de correlación distintos y 15 asociaciones V de Cramér.

Los resultados repetidos entre tablas se deduplican, conservando el vínculo con cada fuente, y las cantidades se verifican contra la línea base. Una técnica no se aplica mecánicamente donde no es válida: se registra el método, el resultado o el motivo de no aplicabilidad. Por ejemplo, los IC *bootstrap* de valores extremos o de variables constantes requieren advertencia, y la V de Cramér es una asociación no negativa, no una correlación con signo.

### Desarrollo completo en anexos

| Anexo | Contenido |
|---|---|
| A | Todos los parámetros y sus IC |
| B | Las siete pruebas, sus distribuciones nulas y Holm |
| C | 20 correlaciones, 15 asociaciones y atlas de distribuciones |
| D | Todos los experimentos de Monte Carlo |
| E | Sensibilidad, jackknife e IDs influyentes |
| F | Matriz completa de resultados para S3 |

La selección de lo que va al cuerpo se justifica en `matriz_editorial.csv` (ID, pregunta, aporte, ubicación y criterio de rúbrica), sin elegir por valor *p*, signo favorable o facilidad de representación.

**Decisiones de alcance.** No se introduce otro conjunto de datos ni se ajusta un modelo predictivo nuevo.

**Limitaciones.** La inferencia sigue condicionada a la independencia y la representatividad de los registros, que el origen del conjunto de datos no permite asegurar. El remuestreo no corrige sesgos de selección ni demuestra causalidad.

### Inventario de resultados de S1 y manifiesto

**Responsable:** Yerko · **Revisor:** José

Se recorren todas las tablas de `sumativa_1/outputs/tablas/` y se registra cada resultado publicado. Los resultados repetidos entre tablas se deduplican, comprobando que su valor y su n coincidan, y se conserva el vínculo con cada fuente. Cada resultado queda con su ID, familia, variable o grupo, n, estimación, IC original (o «no disponible en S1») y método de S2, o el motivo por el que la técnica no se aplica. Los conteos se vinculan a su proporción mediante n; un n fijo no se remuestrea.

`docs/manifiesto_s1.json` congela las entradas de S1: commit, rutas y huellas SHA-256 de cada archivo, fuente de cada parámetro y matriz de cobertura. Se crea una sola vez; en las ejecuciones siguientes solo se verifica.

In [ ]:
TABLAS_S1 = RAIZ_S1 / 'outputs/tablas'
TABLAS_S2 = RAIZ / 'outputs/tablas'
COMMIT_S1 = '04e6df05454a5d12e73b174f022313b20fd80032'   # línea base de la planificación


def huella(ruta):
    """SHA-256 con saltos de línea normalizados a LF, para que coincida en cualquier sistema operativo."""
    return hashlib.sha256(ruta.read_bytes().replace(b'\r\n', b'\n')).hexdigest()


# Familia de S1 a la que pertenece cada tabla
FAMILIA_TABLA = {
    'estimaciones.csv': 'parametros', 'intervalos_confianza.csv': 'parametros',
    'supuestos_intervalos.csv': 'descriptivas', 'descriptiva_cuantitativas.csv': 'descriptivas',
    'frecuencias.csv': 'descriptivas', 'segmento_descuento.csv': 'descriptivas',
    'relacion_dependiente_cuantitativas.csv': 'descriptivas',
    'contraste_peso.csv': 'contrastes', 'contraste_descuento.csv': 'contrastes',
    'contraste_costo.csv': 'contrastes', 'contraste_modalidad.csv': 'contrastes',
    'contrastes_complementarios.csv': 'contrastes', 'contrastes_consolidados.csv': 'contrastes',
    'contrastes_holm_7.csv': 'contrastes', 'comparaciones_proporciones.csv': 'contrastes',
    'correlacion_spearman.csv': 'correlaciones', 'correlacion_dependiente_cuantitativas.csv': 'correlaciones',
    'asociacion_cramer.csv': 'asociaciones', 'relacion_dependiente_cualitativas.csv': 'asociaciones',
    'calidad_datos.csv': 'control de calidad (sin estimandos)',
    'entorno_ejecucion.json': 'entorno (sin estimandos)',
}
archivos_s1 = sorted(p.name for p in TABLAS_S1.iterdir() if p.suffix in {'.csv', '.json'})
assert set(archivos_s1) == set(FAMILIA_TABLA), \
    f'Tablas de S1 sin clasificar o faltantes: {set(archivos_s1) ^ set(FAMILIA_TABLA)}'

huellas_s1 = {nombre: huella(TABLAS_S1 / nombre) for nombre in archivos_s1}
print(f'{len(huellas_s1)} archivos de S1 con huella SHA-256.')

In [ ]:
filas = []

def agregar(fuente, columna, estimando, variable, grupo, valor, n_res, esquema,
            ic_inf=None, ic_sup=None, metodo_ic=None):
    """Registra un resultado publicado en S1 (una fila por aparición en una tabla)."""
    filas.append(dict(fuente=fuente, columna=columna, estimando=estimando, variable=variable,
                      grupo=grupo, n=str(n_res), estimacion_s1=float(valor), esquema=esquema,
                      ic_inf_s1=ic_inf, ic_sup_s1=ic_sup, metodo_ic_s1=metodo_ic))

def leer(nombre, **kw):
    return pd.read_csv(TABLAS_S1 / nombre, float_precision='round_trip', **kw)

def par(a, b):
    """Nombre canónico de un par de variables (orden alfabético; el atraso siempre al final)."""
    if TARGET in (a, b):
        return f'{a if b == TARGET else b}|{TARGET}'
    return '|'.join(sorted([a, b]))

ATR, PUNT, DIF = 'atrasada', 'puntual', 'atrasada - puntual'
N_DIF = f'{n_atr}/{n_punt}'          # n de los contrastes atrasadas - puntuales
VARIABLE_PRUEBA = {'T01': 'modalidad_envio', 'T02': 'peso_gramos', 'T03': 'descuento_ofrecido',
                   'E1': 'peso_gramos', 'E2': 'importancia_producto', 'E3': 'descuento_ofrecido',
                   'E4': 'costo_producto'}
GRUPO_CANONICO = {'importancia alta': 'importancia_producto=high',
                  'resto': 'importancia_producto en {low, medium}',
                  'descuento > 10': 'descuento_ofrecido > 10',
                  'descuento <= 10': 'descuento_ofrecido <= 10'}

# 1) Parámetros principales (estimaciones.csv e intervalos_confianza.csv)
for fuente in ['estimaciones.csv', 'intervalos_confianza.csv']:
    for _, r in leer(fuente).iterrows():
        es_prop = r['variable'] == TARGET
        agregar(fuente, 'estimacion', 'proporcion' if es_prop else 'media', r['variable'],
                '1' if es_prop else 'global', r['estimacion'], r['n'], 'global',
                r['ic_inf'], r['ic_sup'], r['metodo'])

# 2) Descriptivas globales
DESCRIPTIVAS = {'mean': 'media', 'std': 'de', 'min': 'minimo', '25%': 'q25', '50%': 'mediana',
                '75%': 'q75', 'max': 'maximo', 'IQR': 'iqr', 'asimetria': 'asimetria',
                'curtosis_exceso': 'curtosis_exceso'}
for var, r in leer('descriptiva_cuantitativas.csv', index_col=0).iterrows():
    for col, est in DESCRIPTIVAS.items():
        agregar('descriptiva_cuantitativas.csv', col, est, var, 'global', r[col], int(r['count']), 'global')

SUPUESTOS = {'asimetria': 'asimetria', 'curtosis_exceso': 'curtosis_exceso', 'media': 'media',
             'mediana': 'mediana', 'extremos_IQR': 'n_extremos_iqr', '%_suma_top1%': 'pct_suma_top1'}
for _, r in leer('supuestos_intervalos.csv').iterrows():
    for col, est in SUPUESTOS.items():
        agregar('supuestos_intervalos.csv', col, est, r['variable'], 'global', r[col], r['n_efectivo'], 'global')

# Conteos de frecuencias: se registran como proporción; el conteo se recupera con proporción x n
for _, r in leer('frecuencias.csv').iterrows():
    agregar('frecuencias.csv', 'porcentaje', 'proporcion', r['variable'], str(r['categoria']),
            r['porcentaje'] / 100, n, 'global')

# 3) Descriptivas por grupo de atraso
for _, r in leer('relacion_dependiente_cuantitativas.csv').iterrows():
    v = r['variable']
    fuente = 'relacion_dependiente_cuantitativas.csv'
    agregar(fuente, 'media_atrasada', 'media', v, ATR, r['media_atrasada'], n_atr, 'grupo')
    agregar(fuente, 'media_a_tiempo', 'media', v, PUNT, r['media_a_tiempo'], n_punt, 'grupo')
    agregar(fuente, 'mediana_atrasada', 'mediana', v, ATR, r['mediana_atrasada'], n_atr, 'grupo')
    agregar(fuente, 'mediana_a_tiempo', 'mediana', v, PUNT, r['mediana_a_tiempo'], n_punt, 'grupo')
    agregar(fuente, 'd_av', 'd_av', v, DIF, r['d_av'], N_DIF, 'grupo')

for _, r in leer('segmento_descuento.csv').iterrows():
    grupo = GRUPO_CANONICO['descuento > 10' if r['descuento_mayor_10'] else 'descuento <= 10']
    agregar('segmento_descuento.csv', 'mean', 'tasa_atraso', 'descuento_ofrecido', grupo, r['mean'],
            r['size'], 'grupo', r['ic_wilson_inf'], r['ic_wilson_sup'], 'Wilson (score)')

# Tasas mínima y máxima por variable: S1 no indica la categoría, se identifica con los datos
for _, r in leer('relacion_dependiente_cualitativas.csv').iterrows():
    v = r['variable']
    tasas = df.groupby(v, observed=True)[TARGET].agg(['mean', 'size'])
    for col in ['tasa_min_%', 'tasa_max_%']:
        cat = (tasas['mean'] * 100 - r[col]).abs().idxmin()
        agregar('relacion_dependiente_cualitativas.csv', col, 'tasa_atraso', v, f'{v}={cat}',
                r[col] / 100, tasas.loc[cat, 'size'], 'grupo')
    agregar('relacion_dependiente_cualitativas.csv', 'V_cramer', 'v_cramer', par(v, TARGET),
            'global', r['V_cramer'], n, 'global')

# 4) Contrastes de medias (T02, T03, E4)
for fuente in ['contraste_peso.csv', 'contraste_descuento.csv', 'contraste_costo.csv']:
    r = leer(fuente).iloc[0]
    v, cod, n_g = r['variable'], r['codigo'], f"{r['n_atr']}/{r['n_punt']}"
    agregar(fuente, 'media_atr', 'media', v, ATR, r['media_atr'], r['n_atr'], 'grupo')
    agregar(fuente, 'media_punt', 'media', v, PUNT, r['media_punt'], r['n_punt'], 'grupo')
    agregar(fuente, 'de_atr', 'de', v, ATR, r['de_atr'], r['n_atr'], 'grupo')
    agregar(fuente, 'de_punt', 'de', v, PUNT, r['de_punt'], r['n_punt'], 'grupo')
    agregar(fuente, 'diferencia', 'diferencia_medias', v, DIF, r['diferencia'], n_g, 'grupo',
            r['ic_inf'], r['ic_sup'], 't de Welch')
    agregar(fuente, 'd_av', 'd_av', v, DIF, r['d_av'], n_g, 'grupo')
    agregar(fuente, 't', 'estadistico', v, cod, r['t'], n_g, 'prueba')
    agregar(fuente, 'p', 'p', v, cod, r['p'], n_g, 'prueba')

# 5) Chi-cuadrado (T01)
r = leer('contraste_modalidad.csv').iloc[0]
agregar('contraste_modalidad.csv', 'chi2', 'estadistico', r['variable'], 'T01', r['chi2'], r['n'], 'prueba')
agregar('contraste_modalidad.csv', 'p', 'p', r['variable'], 'T01', r['p'], r['n'], 'prueba')
agregar('contraste_modalidad.csv', 'v_cramer', 'v_cramer', par(r['variable'], TARGET), 'global',
        r['v_cramer'], r['n'], 'global')

# 6) Comparaciones de proporciones (E2, E3)
comparaciones = leer('comparaciones_proporciones.csv').set_index('codigo')
for cod, r in comparaciones.iterrows():
    v, n_g = VARIABLE_PRUEBA[cod], f"{r['n1']}/{r['n0']}"
    g1, g0 = GRUPO_CANONICO[r['grupo1']], GRUPO_CANONICO[r['grupo0']]
    fuente = 'comparaciones_proporciones.csv'
    agregar(fuente, 'p1', 'tasa_atraso', v, g1, r['p1'], r['n1'], 'grupo')
    agregar(fuente, 'p0', 'tasa_atraso', v, g0, r['p0'], r['n0'], 'grupo')
    agregar(fuente, 'diferencia', 'diferencia_proporciones', v, f'{g1} - {g0}', r['diferencia'], n_g,
            'grupo', r['ic_inf'], r['ic_sup'], 'Wald (dos proporciones)')
    agregar(fuente, 'riesgo_relativo', 'riesgo_relativo', v, f'{g1} / {g0}', r['riesgo_relativo'], n_g, 'grupo')
    agregar(fuente, 'z', 'estadistico', v, cod, r['z'], n_g, 'prueba')
    agregar(fuente, 'p', 'p', v, cod, r['p'], n_g, 'prueba')

# 7) Contrastes complementarios (E1-E4) y Holm
def n_prueba(cod):
    return f"{comparaciones.loc[cod, 'n1']}/{comparaciones.loc[cod, 'n0']}" if cod in comparaciones.index else (
        n if cod == 'T01' else N_DIF)

for _, r in leer('contrastes_complementarios.csv').iterrows():
    cod = r['codigo']
    v, n_g = VARIABLE_PRUEBA[cod], n_prueba(cod)
    fuente = 'contrastes_complementarios.csv'
    agregar(fuente, 'valor_estadistico', 'estadistico', v, cod, r['valor_estadistico'], n_g, 'prueba')
    agregar(fuente, 'p', 'p', v, cod, r['p'], n_g, 'prueba')
    if r['efecto'] == 'r biserial de rangos':
        agregar(fuente, 'valor_efecto', 'r_biserial_rangos', v, DIF, r['valor_efecto'], n_g, 'grupo')
    elif r['efecto'] == 'd_av':
        agregar(fuente, 'valor_efecto', 'd_av', v, DIF, r['valor_efecto'], n_g, 'grupo')
    else:  # diferencia de proporciones
        c = comparaciones.loc[cod]
        grupos = f"{GRUPO_CANONICO[c['grupo1']]} - {GRUPO_CANONICO[c['grupo0']]}"
        agregar(fuente, 'valor_efecto', 'diferencia_proporciones', v, grupos, r['valor_efecto'], n_g, 'grupo')

for fuente in ['contrastes_consolidados.csv', 'contrastes_holm_7.csv']:
    for _, r in leer(fuente).iterrows():
        cod = r['codigo']
        v, n_g = VARIABLE_PRUEBA[cod], n_prueba(cod)
        agregar(fuente, 'p', 'p', v, cod, r['p'], n_g, 'prueba')
        agregar(fuente, 'p_holm', 'p_holm', v, cod, r['p_holm'], n_g, 'prueba')

# 8) Correlaciones (20) y asociaciones (15): solo pares únicos, sin diagonal
for fuente, est in [('correlacion_spearman.csv', 'rho_spearman'), ('asociacion_cramer.csv', 'v_cramer')]:
    m = leer(fuente, index_col=0)
    for i, a in enumerate(m.index):
        for b in m.columns[i + 1:]:
            agregar(fuente, b, est, par(a, b), 'global', m.loc[a, b], n, 'global')

for _, r in leer('correlacion_dependiente_cuantitativas.csv').iterrows():
    nombre_par = par(r['variable'], TARGET)
    fuente = 'correlacion_dependiente_cuantitativas.csv'
    agregar(fuente, 'r_punto_biserial', 'r_punto_biserial', nombre_par, 'global', r['r_punto_biserial'], n, 'global')
    agregar(fuente, 'rho_spearman', 'rho_spearman', nombre_par, 'global', r['rho_spearman'], n, 'global')
    agregar(fuente, 'p_punto_biserial', 'p_correlacion', nombre_par, 'r_punto_biserial', r['p_punto_biserial'], n, 'global')
    agregar(fuente, 'p_spearman', 'p_correlacion', nombre_par, 'rho_spearman', r['p_spearman'], n, 'global')

apariciones = pd.DataFrame(filas)
print(f'{len(apariciones)} apariciones de resultados en {apariciones["fuente"].nunique()} tablas de S1.')

In [ ]:
CLAVE = ['estimando', 'variable', 'grupo']

def consolidar(g):
    """Une las apariciones repetidas de un resultado y comprueba que su valor y su n coincidan."""
    valores = g['estimacion_s1']
    assert valores.max() - valores.min() <= 1e-9 * max(1.0, valores.abs().max()), \
        f'Valores distintos para el mismo resultado: {g.name}'
    assert g['n'].nunique() == 1, f'n distintos para el mismo resultado: {g.name}'
    con_ic = g.dropna(subset=['ic_inf_s1'])
    ic = con_ic.iloc[0] if len(con_ic) else None
    return pd.Series({
        'n': g['n'].iloc[0],
        'estimacion_s1': valores.iloc[0],
        'ic_inf_s1': ic['ic_inf_s1'] if ic is not None else None,
        'ic_sup_s1': ic['ic_sup_s1'] if ic is not None else None,
        'metodo_ic_s1': ic['metodo_ic_s1'] if ic is not None else 'no disponible en S1',
        'esquema': g['esquema'].iloc[0],
        'fuentes': '; '.join(sorted(set(g['fuente'] + ':' + g['columna']))),
    })

inventario = (apariciones.groupby(CLAVE, sort=False)[apariciones.columns.drop(CLAVE)]
              .apply(consolidar).reset_index())

# Familia de cada resultado
PRINCIPALES = {('proporcion', TARGET, '1'), ('media', 'peso_gramos', 'global'),
               ('media', 'costo_producto', 'global'), ('media', 'llamadas_atencion', 'global')}
FAMILIA = {'diferencia_medias': 'efectos', 'd_av': 'efectos', 'diferencia_proporciones': 'efectos',
           'riesgo_relativo': 'efectos', 'r_biserial_rangos': 'efectos',
           'estadistico': 'pruebas', 'p': 'pruebas', 'p_holm': 'pruebas',
           'rho_spearman': 'correlaciones', 'r_punto_biserial': 'correlaciones',
           'p_correlacion': 'correlaciones', 'v_cramer': 'asociaciones'}
inventario['familia'] = [
    'parametros' if c in PRINCIPALES else FAMILIA.get(c[0], 'descriptivas')
    for c in zip(inventario['estimando'], inventario['variable'], inventario['grupo'])]

# Método de S2, o motivo por el que no se aplica la técnica
def metodo_s2(r):
    if r['estimando'] == 'estadistico':
        return 'Permutación (S01): estadístico recalculado en cada permutación'
    if r['estimando'] == 'p':
        return 'Permutación (S01): p de permutación'
    if r['estimando'] == 'p_holm':
        return 'Permutación (S01): Holm sobre los 7 p de permutación'
    if r['estimando'] == 'p_correlacion':
        return 'Bootstrap de coeficientes (S02): no se recalcula el p; se evalúa con el IC del coeficiente'
    if r['familia'] in ('correlaciones', 'asociaciones'):
        return 'Bootstrap de coeficientes (S02): filas emparejadas, IC percentil'
    if r['estimando'] in ('minimo', 'maximo'):
        return 'Bootstrap (Y02): solo diagnóstico; el bootstrap de extremos no tiene cobertura poblacional'
    if r['esquema'] == 'grupo':
        return 'Bootstrap (Y02): remuestreo dentro de cada grupo, IC percentil y BCa'
    return 'Bootstrap (Y02): remuestreo de filas, IC percentil y BCa'

inventario['metodo_s2'] = inventario.apply(metodo_s2, axis=1)

# Identificadores ordenados por familia
ORDEN = ['parametros', 'efectos', 'descriptivas', 'pruebas', 'correlaciones', 'asociaciones']
PREFIJO = dict(zip(ORDEN, ['PAR', 'EFE', 'DES', 'PRU', 'COR', 'ASO']))
inventario['familia'] = pd.Categorical(inventario['familia'], categories=ORDEN, ordered=True)
inventario = inventario.sort_values(['familia', 'estimando', 'variable', 'grupo'], ignore_index=True)
inventario['id'] = [f'{PREFIJO[f]}-{i:03d}' for f, i in
                    zip(inventario['familia'], inventario.groupby('familia', observed=True).cumcount() + 1)]
inventario = inventario[['id', 'familia', 'estimando', 'variable', 'grupo', 'n', 'estimacion_s1',
                         'ic_inf_s1', 'ic_sup_s1', 'metodo_ic_s1', 'fuentes', 'metodo_s2']]

# Verificación de cantidades contra la línea base de la planificación
cuenta = inventario['estimando'].value_counts()
assert (inventario['familia'] == 'parametros').sum() == 4, 'Deben ser 4 parámetros principales.'
assert cuenta['estadistico'] == cuenta['p'] == cuenta['p_holm'] == 7, 'Deben ser 7 pruebas.'
assert cuenta['rho_spearman'] + cuenta['r_punto_biserial'] == 20, 'Deben ser 20 correlaciones.'
assert cuenta['v_cramer'] == 15, 'Deben ser 15 asociaciones V de Cramér.'

inventario.to_csv(TABLAS_S2 / 'inventario_resultados_s1.csv', index=False)
print(f'Inventario: {len(apariciones)} apariciones -> {len(inventario)} resultados únicos.')

# Matriz de cobertura: resultados de cada familia por método de S2
cobertura = pd.crosstab(inventario['metodo_s2'], inventario['familia'], margins=True, margins_name='Total')
cobertura

In [ ]:
RUTA_MANIFIESTO = RAIZ / 'docs/manifiesto_s1.json'

if RUTA_MANIFIESTO.is_file() and RUTA_MANIFIESTO.stat().st_size > 0:
    manifiesto = json.loads(RUTA_MANIFIESTO.read_text(encoding='utf-8'))
    distintas = [t['archivo'] for t in manifiesto['tablas'] if huellas_s1.get(t['archivo']) != t['sha256']]
    assert not distintas, f'Tablas de S1 modificadas desde el congelamiento: {distintas}'
    print(f"Manifiesto verificado: {len(manifiesto['tablas'])} archivos de S1 sin cambios.")
else:
    manifiesto = {
        'commit_s1': COMMIT_S1,
        'datos': {'ruta': DATOS.relative_to(RAIZ.parent).as_posix(), 'sha256': HUELLA_DATOS},
        'tablas': [{'archivo': a, 'ruta': (TABLAS_S1 / a).relative_to(RAIZ.parent).as_posix(),
                    'familia': FAMILIA_TABLA[a], 'sha256': huellas_s1[a]} for a in archivos_s1],
        'parametros_principales': [
            {'parametro': r['parametro'], 'variable': r['variable'], 'estimacion': r['estimacion'],
             'ic_95': [r['ic_inf'], r['ic_sup']], 'metodo': r['metodo'],
             'fuente': (TABLAS_S1 / 'estimaciones.csv').relative_to(RAIZ.parent).as_posix()}
            for _, r in leer('estimaciones.csv').iterrows()],
        'matriz_cobertura': {
            'inventario': (TABLAS_S2 / 'inventario_resultados_s1.csv').relative_to(RAIZ.parent).as_posix(),
            'resultados': len(inventario),
            'por_familia_y_metodo': {m: {f: int(c) for f, c in fila.items() if c}
                                     for m, fila in cobertura.drop(index='Total', columns='Total').iterrows()},
        },
    }
    RUTA_MANIFIESTO.write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Manifiesto creado: {len(huellas_s1)} archivos de S1, commit {COMMIT_S1[:7]}.')

## 2. Bootstrap y pruebas de permutación

### 2.1 Bootstrap de todos los parámetros y efectos

**Responsable:** Yerko · **Revisor:** Sebastián · **Desarrollo completo:** Anexo A

**Método.** Se procesa el inventario completo con **10.000 remuestras por resultado** e IC percentil y BCa al 95 %, conservando el estimador, la unidad, la población y los grupos de S1:

- **Parámetros globales:** se remuestrean filas completas, de tamaño 10.999.
- **Diferencias y efectos entre grupos:** se remuestrea dentro de cada grupo, manteniendo sus tamaños.
- Las réplicas se comparten dentro de una misma familia cuando corresponde.

| Familia de S1 | Cobertura y comparación |
|---|---|
| `estimaciones.csv` / `intervalos_confianza.csv` | Los cuatro parámetros (atraso, peso, costo y llamadas), comparados con el IC de Wilson o *t* original |
| Contrastes y `comparaciones_proporciones.csv` | Diferencias de peso, descuento y costo; diferencias de proporciones de E2 y E3; efectos $d_{av}$, riesgo relativo y biserial de rangos |
| Descriptivas, frecuencias y tasas por grupo | Todas las medias, medianas, dispersiones, cuantiles, proporciones y tasas publicadas, identificando cuáles no tenían IC clásico |
| Extremos, constantes y fronteras | Mínimo y máximo como diagnóstico descriptivo, sin atribuir cobertura poblacional al *bootstrap* ordinario de extremos; se registran las degeneraciones y los BCa no estimables |

**Inventario.** `inventario_resultados_s1.csv` registra ID, fuente, variable o grupo, estimación, IC original y método de S2. Si no hay IC clásico, se marca «no disponible en S1». Los conteos y proporciones redundantes se vinculan mediante n; un n fijo no se remuestrea.

**BCa.** Trata empates y estima la aceleración por *jackknife*; se valida contra una implementación de referencia y se procesa por lotes. No se sustituye *t* por *z* ni se supone robustez automática; se conservan las unidades y los estimandos.

**Evidencias:** `outputs/tablas/bootstrap_intervalos.csv` (una fila por estimando: n/B, estimación de S1, sesgo, EE, $z_0$, aceleración, IC percentil y BCa, y diferencias de extremos y ancho), distribuciones de todos los estimandos y un atlas en el Anexo A.

**Para el cuerpo del informe** *(pendiente)*: al menos tres parámetros relevantes, cada uno con IC clásico, percentil y BCa, su distribución e interpretación crítica. Propuesta: proporción de atraso (riesgo operativo), peso medio (perfil logístico) y costo medio (perfil económico); las llamadas solo si aportan una conclusión necesaria sobre atención.

#### Estimadores y bloques de remuestreo

Cada estimador replica exactamente el de S1 (por ejemplo, asimetría y curtosis corregidas por sesgo, cuartiles con interpolación lineal y la regla 1,5·IQR) y se calcula de forma vectorizada sobre lotes de remuestras. Los resultados se agrupan en **bloques** que comparten remuestras:

| Bloque | Muestras que se remuestrean | Resultados |
|---|---|---|
| Descriptivas globales de cada variable cuantitativa | Filas completas | Media, DE, mínimo, cuartiles, máximo, IQR, asimetría, curtosis, extremos IQR y % de suma del top 1 % |
| Proporciones de cada variable categórica | Filas completas | Proporción de cada categoría (incluye la proporción de atraso) |
| Cada variable cuantitativa por atraso | Atrasadas y puntuales, por separado | Media, mediana y DE por grupo; diferencia de medias y $d_{av}$ |
| Biserial de rangos (E1) | Atrasadas y puntuales | $r = 2U/(n_1 n_0) - 1$, con rangos promedio ante empates |
| Tasa de atraso por categoría | Cada categoría, por separado | Tasa de atraso de cada categoría |
| Comparaciones E2 y E3 | Los dos grupos de cada comparación | Tasas, diferencia de proporciones y riesgo relativo |

Antes de remuestrear, cada estimador se evalúa sobre los datos originales y se exige que coincida con el valor de S1.

In [ ]:
from scipy.stats import rankdata

inventario_s1 = pd.read_csv(TABLAS_S2 / 'inventario_resultados_s1.csv', dtype={'grupo': str, 'n': str},
                            float_precision='round_trip')
inv_y02 = inventario_s1[inventario_s1['metodo_s2'].str.startswith('Bootstrap (Y02)')]
claves_y02 = set(zip(inv_y02['estimando'], inv_y02['variable'], inv_y02['grupo']))
CUANTITATIVAS = ['llamadas_atencion', 'costo_producto', 'compras_previas', 'descuento_ofrecido', 'peso_gramos']
CATEGORICAS = ['bloque_bodega', 'modalidad_envio', 'calificacion_cliente', 'importancia_producto', 'genero']
y = df[TARGET].to_numpy()


def asimetria(m):
    """Asimetría corregida por sesgo (G1), como stats.skew(bias=False) de S1."""
    n = m.shape[1]
    d = m - m.mean(axis=1, keepdims=True)
    m2, m3 = (d ** 2).mean(axis=1), (d ** 3).mean(axis=1)
    return m3 / m2 ** 1.5 * np.sqrt(n * (n - 1)) / (n - 2)


def curtosis(m):
    """Curtosis en exceso corregida por sesgo (G2), como stats.kurtosis(bias=False) de S1."""
    n = m.shape[1]
    d = m - m.mean(axis=1, keepdims=True)
    m2, m4 = (d ** 2).mean(axis=1), (d ** 4).mean(axis=1)
    g2 = m4 / m2 ** 2 - 3
    return ((n + 1) * g2 + 6) * (n - 1) / ((n - 2) * (n - 3))


def descriptivas_globales(m):
    q25, q50, q75 = np.quantile(m, [0.25, 0.5, 0.75], axis=1)
    iqr = q75 - q25
    extremos = ((m < (q25 - 1.5 * iqr)[:, None]) | (m > (q75 + 1.5 * iqr)[:, None])).sum(axis=1)
    k = max(1, m.shape[1] // 100)                                   # top 1 %, como nlargest de S1
    top1 = -np.partition(-m, k - 1, axis=1)[:, :k].sum(axis=1) / m.sum(axis=1) * 100
    return np.column_stack([m.mean(axis=1), m.std(axis=1, ddof=1), m.min(axis=1), q25, q50, q75,
                            m.max(axis=1), iqr, asimetria(m), curtosis(m), extremos, top1])


def por_atraso(a, p):
    va, vp = a.var(axis=1, ddof=1), p.var(axis=1, ddof=1)
    dif = a.mean(axis=1) - p.mean(axis=1)
    return np.column_stack([a.mean(axis=1), p.mean(axis=1), np.median(a, axis=1), np.median(p, axis=1),
                            np.sqrt(va), np.sqrt(vp), dif, dif / np.sqrt((va + vp) / 2)])


def biserial_rangos(a, p):
    n1, n0 = a.shape[1], p.shape[1]
    rangos = rankdata(np.concatenate([a, p], axis=1), axis=1)       # rangos promedio ante empates
    u = rangos[:, :n1].sum(axis=1) - n1 * (n1 + 1) / 2
    return 2 * u / (n1 * n0) - 1


def tasas(*grupos):
    return np.column_stack([g.mean(axis=1) for g in grupos])


def comparacion(a, p):
    p1, p0 = a.mean(axis=1), p.mean(axis=1)
    return np.column_stack([p1, p0, p1 - p0, p1 / p0])


# Bloques: (nombre, estadístico, muestras, claves de los resultados que produce)
bloques = []
for v in CUANTITATIVAS:
    x = df[v].to_numpy(dtype=float)
    bloques.append((f'globales {v}', descriptivas_globales, (x,),
                    [(e, v, 'global') for e in ['media', 'de', 'minimo', 'q25', 'mediana', 'q75', 'maximo',
                                                'iqr', 'asimetria', 'curtosis_exceso', 'n_extremos_iqr',
                                                'pct_suma_top1']]))
for c in [TARGET] + CATEGORICAS:
    codigos, categorias = pd.factorize(df[c].astype(str), sort=True)
    bloques.append((f'proporciones {c}', lambda m, k=len(categorias): np.column_stack([(m == j).mean(axis=1) for j in range(k)]),
                    (codigos,), [('proporcion', c, cat) for cat in categorias]))
for v in CUANTITATIVAS:
    x = df[v].to_numpy(dtype=float)
    bloques.append((f'por atraso {v}', por_atraso, (x[y == 1], x[y == 0]),
                    [('media', v, ATR), ('media', v, PUNT), ('mediana', v, ATR), ('mediana', v, PUNT),
                     ('de', v, ATR), ('de', v, PUNT), ('diferencia_medias', v, DIF), ('d_av', v, DIF)]))
x = df['peso_gramos'].to_numpy(dtype=float)
bloques.append(('biserial E1', biserial_rangos, (x[y == 1], x[y == 0]),
                [('r_biserial_rangos', 'peso_gramos', DIF)]))
for c in CATEGORICAS:
    categorias = sorted(df[c].astype(str).unique())
    bloques.append((f'tasas {c}', tasas, tuple(y[df[c].astype(str) == cat] for cat in categorias),
                    [('tasa_atraso', c, f'{c}={cat}') for cat in categorias]))
for v, en_g1 in [('importancia_producto', df['importancia_producto'] == 'high'),
                 ('descuento_ofrecido', df['descuento_ofrecido'] > 10)]:
    g1, g0 = [GRUPO_CANONICO[k] for k in (['importancia alta', 'resto'] if v == 'importancia_producto'
                                          else ['descuento > 10', 'descuento <= 10'])]
    en_g1 = en_g1.to_numpy()
    bloques.append((f'comparación {v}', comparacion, (y[en_g1], y[~en_g1]),
                    [('tasa_atraso', v, g1), ('tasa_atraso', v, g0),
                     ('diferencia_proporciones', v, f'{g1} - {g0}'), ('riesgo_relativo', v, f'{g1} / {g0}')]))

# Cada resultado del inventario debe salir de algún bloque, y cada estimador debe reproducir S1
producidas = {c for *_, claves in bloques for c in claves}
assert claves_y02 <= producidas, f'Resultados sin bloque: {claves_y02 - producidas}'
s1 = inv_y02.set_index(['estimando', 'variable', 'grupo'])['estimacion_s1']
for nombre, est, muestras, claves in bloques:
    original = np.atleast_1d(np.asarray(est(*[m[None, :] for m in muestras]))[0])
    for clave, valor in zip(claves, original):
        if clave in s1.index:
            assert np.isclose(valor, s1[clave], rtol=1e-9, atol=1e-12), f'{clave}: {valor} ≠ S1 {s1[clave]}'
print(f'{len(bloques)} bloques; {len(claves_y02)} resultados del inventario; todos los estimadores reproducen S1.')

#### Ejecución del bootstrap (B = 10.000)

Todos los bloques usan el flujo `bootstrap_parametros`, en el orden fijo de la lista. Para cada resultado se registran las réplicas, el *jackknife* (aceleración del BCa) y el resumen de `intervalos_bootstrap()`. Los mínimos y máximos se informan solo como diagnóstico.

In [ ]:
import time

B_BOOT = 10_000
rng = generador('bootstrap_parametros')
REPLICAS_Y02, filas_boot = {}, []
inicio_y02 = time.perf_counter()

for nombre, est, muestras, claves in bloques:
    usadas = [i for i, c in enumerate(claves) if c in claves_y02 and c not in REPLICAS_Y02]
    if not usadas:
        continue
    rep = bootstrap(est, muestras, rng, B=B_BOOT).reshape(B_BOOT, -1)
    jk = jackknife(est, muestras).reshape(-1, rep.shape[1])
    original = np.atleast_1d(np.asarray(est(*[m[None, :] for m in muestras]))[0])
    for i in usadas:
        REPLICAS_Y02[claves[i]] = rep[:, i]
        res = intervalos_bootstrap(rep[:, i], original[i], jk[:, i], tamanos=[len(m) for m in muestras])
        filas_boot.append({'estimando': claves[i][0], 'variable': claves[i][1], 'grupo': claves[i][2], **res})

boot = inv_y02.merge(pd.DataFrame(filas_boot), on=['estimando', 'variable', 'grupo'], validate='one_to_one')
extremo = boot['estimando'].isin(['minimo', 'maximo'])
boot.loc[extremo, 'advertencia'] = ('Extremo muestral: solo diagnóstico, sin cobertura poblacional; '
                                    + boot.loc[extremo, 'advertencia']).str.rstrip('; ')
# Diferencias de extremos y de ancho del BCa respecto del IC clásico de S1 (vacías si S1 no tenía IC)
boot['dif_inf_bca'] = boot['ic_bca_inf'] - boot['ic_inf_s1']
boot['dif_sup_bca'] = boot['ic_bca_sup'] - boot['ic_sup_s1']
boot['dif_ancho_bca'] = (boot['ic_bca_sup'] - boot['ic_bca_inf']) - (boot['ic_sup_s1'] - boot['ic_inf_s1'])

bootstrap_intervalos = boot.rename(columns={'ic_inf_s1': 'ic_clasico_inf', 'ic_sup_s1': 'ic_clasico_sup'})[
    ['id', 'estimando', 'variable', 'grupo', 'n', 'B', 'estimacion_s1', 'sesgo', 'ee', 'z0', 'aceleracion',
     'ic_clasico_inf', 'ic_clasico_sup', 'ic_percentil_inf', 'ic_percentil_sup', 'ic_bca_inf', 'ic_bca_sup',
     'dif_inf_bca', 'dif_sup_bca', 'dif_ancho_bca', 'advertencia']]
assert len(bootstrap_intervalos) == len(inv_y02), 'Faltan resultados del inventario.'
bootstrap_intervalos.to_csv(TABLAS_S2 / 'bootstrap_intervalos.csv', index=False)

print(f'{len(bootstrap_intervalos)} resultados con B = {B_BOOT:,} en {time.perf_counter() - inicio_y02:.0f} s.'.replace(',', '.'))
print(f"BCa no estimable: {bootstrap_intervalos['ic_bca_inf'].isna().sum()} | "
      f"con advertencia: {(bootstrap_intervalos['advertencia'] != '').sum()}")
bootstrap_intervalos[bootstrap_intervalos['id'].str.startswith('PAR')][
    ['id', 'estimando', 'variable', 'estimacion_s1', 'ic_clasico_inf', 'ic_clasico_sup',
     'ic_percentil_inf', 'ic_percentil_sup', 'ic_bca_inf', 'ic_bca_sup']]

#### Validación del BCa contra una implementación de referencia

Se entregan a `scipy.stats.bootstrap` las mismas réplicas (`n_resamples=0`), para que calcule el BCa con su propio *jackknife*. Se comparan una proporción, una media y un efecto entre dos grupos.

In [ ]:
from scipy import stats


def bca_scipy(est, muestras, replicas):
    """BCa de scipy con las mismas réplicas: solo cambia la implementación del jackknife y del BCa."""
    def estadistico(*m, axis=-1):
        m = [np.moveaxis(x, axis, -1) for x in m]
        forma = m[0].shape[:-1]
        return np.asarray(est(*[x.reshape(-1, x.shape[-1]) for x in m])).reshape(forma)
    previo = type('Previo', (), {'bootstrap_distribution': replicas})()
    ic = stats.bootstrap(muestras, estadistico, n_resamples=0, method='BCa', vectorized=True,
                         bootstrap_result=previo).confidence_interval
    return ic.low, ic.high


casos = [(('proporcion', TARGET, '1'), lambda m: m.mean(axis=1), (y,)),
         (('media', 'peso_gramos', 'global'), lambda m: m.mean(axis=1), (df['peso_gramos'].to_numpy(dtype=float),)),
         (('d_av', 'peso_gramos', DIF), lambda a, p: por_atraso(a, p)[:, 7],
          (df.loc[df[TARGET] == 1, 'peso_gramos'].to_numpy(dtype=float),
           df.loc[df[TARGET] == 0, 'peso_gramos'].to_numpy(dtype=float)))]
filas_val = []
for clave, est, muestras in casos:
    propio = bootstrap_intervalos.set_index(['estimando', 'variable', 'grupo']).loc[clave]
    inf, sup = bca_scipy(est, muestras, REPLICAS_Y02[clave])
    filas_val.append({'id': propio['id'], 'estimando': clave[0], 'variable': clave[1],
                      'bca_propio_inf': propio['ic_bca_inf'], 'bca_scipy_inf': inf,
                      'bca_propio_sup': propio['ic_bca_sup'], 'bca_scipy_sup': sup})
validacion_bca = pd.DataFrame(filas_val)
assert np.allclose(validacion_bca[['bca_propio_inf', 'bca_propio_sup']].to_numpy(),
                   validacion_bca[['bca_scipy_inf', 'bca_scipy_sup']].to_numpy(), rtol=1e-10), 'BCa distinto de scipy.'
print('BCa propio = BCa de scipy en los tres casos.')
validacion_bca

#### Distribuciones bootstrap de los parámetros del cuerpo

Proporción de atraso, peso medio y costo medio: distribución de las 10.000 réplicas, estimación de S1 e IC clásico, percentil y BCa.

In [ ]:
from matplotlib.lines import Line2D

PARAMETROS_CUERPO = [(('proporcion', TARGET, '1'), 'Proporción de atraso', 100, '%', 1),
                     (('media', 'peso_gramos', 'global'), 'Peso medio', 1, 'g', 0),
                     (('media', 'costo_producto', 'global'), 'Costo medio', 1, 'USD', 1)]
ESTILO_IC = [('Clásico (S1)', 'ic_clasico', GRIS), ('Percentil', 'ic_percentil', AZUL), ('BCa', 'ic_bca', TERRA)]
tabla_boot = bootstrap_intervalos.set_index(['estimando', 'variable', 'grupo'])

fig, ejes, _ = rejilla(3, ncols=3, alto=2.4)
for ax, (clave, titulo, escala, unidad, dec) in zip(ejes, PARAMETROS_CUERPO):
    fila = tabla_boot.loc[clave]
    alturas, _, _ = ax.hist(REPLICAS_Y02[clave] * escala, bins=50, color=AZUL, alpha=0.35, density=True)
    tope = alturas.max()
    ax.axvline(fila['estimacion_s1'] * escala, color='black', lw=1, ls='--')
    for j, (_, prefijo, color) in enumerate(ESTILO_IC):
        y_ic = -tope * (0.12 + 0.12 * j)
        ax.hlines(y_ic, fila[f'{prefijo}_inf'] * escala, fila[f'{prefijo}_sup'] * escala, color=color, lw=2.2)
    ax.set_ylim(-tope * 0.48, tope * 1.08)
    ax.set_yticks([])
    ax.set_title(titulo)
    ax.set_xlabel(f'Réplica bootstrap ({unidad})')
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _, d=dec: num(v, d)))
fig.legend([Line2D([], [], color=c, lw=2.2) for _, _, c in ESTILO_IC] + [Line2D([], [], color='black', lw=1, ls='--')],
           [e for e, _, _ in ESTILO_IC] + ['Estimación S1'], loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.08))
guardar('fig_boot_1_parametros', fig)

#### Atlas del Anexo A

Distribución bootstrap de cada resultado del inventario a cargo de Y02, en páginas de 12 paneles ordenadas por ID, con la estimación de S1 y los IC percentil y BCa.

In [ ]:
POR_PAGINA = 12
ids = bootstrap_intervalos['id'].tolist()
paginas = [ids[i:i + POR_PAGINA] for i in range(0, len(ids), POR_PAGINA)]
tabla_id = bootstrap_intervalos.set_index('id')

for p, ids_pagina in enumerate(paginas, start=1):
    fig, ejes, sobrantes = rejilla(len(ids_pagina), ncols=3, alto=1.75)
    for ax, i in zip(ejes, ids_pagina):
        fila = tabla_id.loc[i]
        clave = (fila['estimando'], fila['variable'], fila['grupo'])
        ax.hist(REPLICAS_Y02[clave], bins=40, color=AZUL, alpha=0.35)
        ax.axvline(fila['estimacion_s1'], color=GRIS, lw=1)
        for prefijo, color, estilo in [('ic_percentil', AZUL, '--'), ('ic_bca', TERRA, ':')]:
            for lado in ('inf', 'sup'):
                if pd.notna(fila[f'{prefijo}_{lado}']):
                    ax.axvline(fila[f'{prefijo}_{lado}'], color=color, lw=1, ls=estilo)
        grupo = '' if fila['grupo'] == 'global' else f" · {fila['grupo']}"
        ax.set_title(f"{i} · {fila['estimando']}\n{ETIQ_CORTA.get(fila['variable'], fila['variable'])}{grupo}",
                     fontsize=7)
        ax.set_yticks([])
        ax.tick_params(axis='x', labelsize=6.5)
    for ax in sobrantes:
        ax.set_visible(False)
    guardar(f'fig_boot_A{p:02d}_atlas', fig, formatos=('pdf',))
    plt.close(fig)
print(f'Atlas del Anexo A: {len(paginas)} páginas, {len(ids)} distribuciones.')

#### Interpretación

**Cobertura.** Los 128 resultados del inventario a cargo de Y02 tienen réplicas, IC percentil y diagnóstico. El BCa es estimable en 91; en los otros 37 queda registrado como no estimable y se informa el IC percentil con su advertencia:

- **23 con réplicas constantes:** medianas, cuartiles e IQR de variables enteras con muchos empates, mínimos y máximos, conteos de extremos iguales a 0 y la tasa de atraso con descuento > 10.
- **14 con *jackknife* sin variación:** medianas y cuartiles que no cambian al retirar un envío, por lo que la aceleración no se puede estimar.

**Parámetros principales.** Para la proporción de atraso y las medias de peso, costo y llamadas, los IC percentil y BCa prácticamente coinciden con el IC clásico de S1: los anchos difieren en menos de 1 % y $|z_0|$ y la aceleración son cercanos a 0, lo que indica distribuciones bootstrap simétricas. Con n = 10.999, la aproximación de Wilson y *t* de S1 queda respaldada, siempre bajo el supuesto de independencia y representatividad de los envíos.

**Efectos entre grupos.** Los 13 IC BCa de diferencias, $d_{av}$, biserial de rangos y riesgo relativo excluyen el valor nulo (0, o 1 para el riesgo relativo), con la misma dirección que en S1. Cuando S1 tiene IC clásico, los anchos difieren de él en menos de 2 %.

**Casos que no se interpretan como certeza.** La tasa de atraso con descuento > 10 vale 1 en todas las réplicas: el IC [1; 1] refleja una frontera muestral, no una certeza poblacional, y se conserva como advertencia el IC de Wilson de S1, [0,998551; 1]. En el costo, los cuartiles y el IQR toman pocos valores distintos y el BCa se desplaza respecto del percentil ($z_0$ hasta 0,70); allí el IC se lee como diagnóstico. Los mínimos y máximos se informan solo como diagnóstico descriptivo.

**Limitación.** El remuestreo cuantifica la variabilidad muestral, pero no corrige sesgos de selección ni demuestra causalidad.

### 2.2 Permutación de las siete pruebas

**Responsable:** Sebastián · **Revisor:** Yerko · **Desarrollo completo:** Anexo B

**Método.** Cada contraste conserva su $H_0/H_1$, sus grupos, su dirección y su estadístico de S1. Se ejecutan **10.000 permutaciones por prueba**, recalculando siempre el estadístico:

| Prueba de S1 | Estadístico y esquema de permutación |
|---|---|
| T01: modalidad–atraso | $\chi^2$ de Pearson; se permuta el atraso entre modalidades, con márgenes fijos |
| T02: peso por atraso | *t* de Welch studentizada; se permutan las etiquetas de atraso |
| T03: descuento por atraso | *t* de Welch studentizada; mismo esquema |
| E1: peso por atraso | *U* de Mann–Whitney con empates; se permutan las etiquetas y se centra en $n_1 n_0/2$ |
| E2: atraso, importancia alta frente al resto | *z* de dos proporciones; se permuta el atraso entre grupos fijos |
| E3: atraso, descuento > 10 frente a ≤ 10 | *z* de dos proporciones; mismo esquema, advirtiendo la frontera $p_1 = 1$ |
| E4: costo por atraso | *t* de Welch studentizada; se permutan las etiquetas de atraso |

**Valor p.** Para *t* y *z* bilaterales se cuentan las excedencias de $|\text{estadístico}|$; para *U* bilateral, la distancia al centro $n_1 n_0/2$; para $\chi^2$, la cola superior:

$$p_{perm} = \frac{1 + \#\{\text{excedencias}\}}{B + 1}$$

Se guardan la fórmula, los empates y la resolución $1/10.001$. Los *p* registrados como 0 en S1 son subdesbordamiento numérico, no probabilidad exactamente nula.

**Multiplicidad.** Se aplica **Holm a la familia completa de los siete *p* de permutación** y se comparan las decisiones con las de la familia de Holm original, separando las comparaciones crudas de las ajustadas. Con B = 10.000 se reconoce una resolución limitada y una posible incertidumbre de decisión cerca del umbral; si hace falta, se amplía B con una regla documentada.

**Supuestos.** La exactitud requiere intercambiabilidad; la *t* de Welch studentizada tiene justificación asintótica bajo heterocedasticidad; Mann–Whitney contrasta distribuciones o rangos, no igualdad de medias.

**Evidencias:** `outputs/tablas/permutaciones_todas.csv` y las siete distribuciones nulas (Anexo B).

**Para el cuerpo del informe** *(pendiente)*: al menos una prueba relevante, T02 (peso por atraso), con $H_0/H_1$, efecto, estadístico observado, distribución nula, *p* original y de permutación, y discusión de supuestos. Se agrega otra solo si aporta una respuesta distinta y útil al problema; la comparación completa y Holm quedan en el Anexo B, conservando para las pruebas reportadas la referencia al ajuste familiar.

## 3. Correlaciones, asociaciones y simulación Monte Carlo

### 3.1 Todas las correlaciones y asociaciones

**Responsable:** Sebastián · **Revisor:** José · **Desarrollo completo:** Anexo C

**Universo completo:**

| Fuente de S1 | Coeficientes |
|---|---|
| `correlacion_spearman.csv` | 10 pares únicos entre cinco variables cuantitativas |
| `correlacion_dependiente_cuantitativas.csv` | 5 punto-biseriales y 5 Spearman con el atraso |
| `asociacion_cramer.csv` | 10 pares únicos entre cinco variables categóricas |
| `relacion_dependiente_cualitativas.csv` | 5 asociaciones V de Cramér con el atraso |

**Resultado esperado:** 20 correlaciones y 15 asociaciones, con estimación de S1, n/B, EE, IC, ancho, diagnóstico y sensibilidad. Las asociaciones repetidas en T01 se validan una vez y se referencian desde ambos apartados.

**Método.** Se remuestrean filas emparejadas **10.000 veces por coeficiente**, con IC percentil al 95 %. Se conserva el tipo de coeficiente y se recalculan los rangos de Spearman en cada remuestra. Se excluyen las diagonales y los pares simétricos duplicados, y nunca se remuestrean las columnas por separado.

**Interpretación** *(pendiente)*.

- **Correlaciones:** inclusión del cero, signo y precisión; distinguir un efecto débil de uno inestable.
- **V de Cramér**, acotada en [0, 1]: no se traslada la lógica del signo. Se informan la magnitud, el ancho y el sesgo del *bootstrap* cerca de cero; la independencia no se resuelve solo con un IC percentil positivo.

**Evidencias:** `outputs/tablas/correlaciones_bootstrap.csv`, `outputs/tablas/asociaciones_bootstrap.csv`, atlas de distribuciones y todos los IC (Anexo C).

**Para el cuerpo del informe** *(pendiente)*: entre tres y cinco correlaciones de mayor relevancia para los atrasos, con distribución, IC 95 %, estabilidad y magnitud. Propuesta inicial: atraso–descuento y atraso–peso (punto-biseriales) y descuento–peso (Spearman). Dos coeficientes del mismo par no cuentan como dos hallazgos, y no se agregan 35 pruebas nuevas a la familia de Holm original.

### 3.2 Monte Carlo y extensión a todos los parámetros

**Responsable:** José · **Revisor:** Yerko · **Desarrollo completo:** Anexo D

**Pregunta.** ¿Qué variabilidad del porcentaje de atrasos cabe esperar en lotes comparables al conjunto de datos, y cómo cambia con el tamaño del lote?

**Modelo.** $K \sim \text{Binomial}(N, p)$ con $p = 6.563/10.999$, la proporción de S1, y tasa $K/N$. Tamaños de lote $N$ = 100, 500, 1.000 y 5.000 (una decisión de escenario, no un parámetro estimado), con **10.000 lotes por tamaño**.

**Resultados por tamaño:** media, desviación estándar, cuantiles 2,5, 97,5 y 99, y $P(\text{tasa} > 0{,}60)$. El 60 % es un umbral ilustrativo, no un objetivo empresarial acreditado.

**Controles internos:** media teórica $p$, varianza $p(1-p)/N$ y probabilidad exacta binomial. La coincidencia demuestra la coherencia del simulador, no una validación externa del modelo.

**Convergencia.** Media y probabilidad acumuladas para B = 100, 500, 1.000, 5.000 y 10.000, con $\text{MCSE}_{media} = s(\text{tasa})/\sqrt{B}$ y $\text{MCSE}_{prob} = \sqrt{\hat q(1-\hat q)/B}$. Tolerancias fijadas antes de ejecutar: $\text{MCSE}_{media} \le 0{,}001$ y $\text{MCSE}_{prob} \le 0{,}005$, junto con la diferencia frente al valor analítico. Si no se cumplen, se amplía hasta 50.000 lotes por escenario y se documenta. Si un evento no ocurre en ninguna simulación, se reporta su límite binomial en vez de afirmar incertidumbre nula.

**Interpretación** *(pendiente)*. Los cuantiles forman un **intervalo de resultados simulados condicionado a $p$**, no un IC del parámetro $p$; la incertidumbre de estimación se examina aparte, variando $p$ dentro del IC de S1 (sección 4.3).

**Extensión a todos los estimandos (Anexo D).** Para cada estimando del inventario se evalúa su comportamiento por Monte Carlo usando remuestras conjuntas de la distribución empírica de S1, con B ≥ 10.000. Las réplicas del *bootstrap* pueden reutilizarse como experimento de Monte Carlo explícitamente identificado, sin contarlas como evidencia independiente. Se reportan media, dispersión, cuantiles, sesgo respecto del valor empírico y MCSE cuando esté definido. Se conservan la dependencia, la estructura de grupos y los dominios: la simulación operativa principal sigue siendo binomial, no se inventan distribuciones paramétricas ni se generan variables independientes para «usar todos los parámetros». La referencia poblacional de esta extensión es la distribución empírica, no una población externa conocida.

**Evidencias:** `outputs/tablas/montecarlo_escenarios.csv`, `outputs/tablas/montecarlo_convergencia.csv` y figuras de distribución y convergencia.

**Para el cuerpo del informe** *(pendiente)*: el lote N = 1.000, su riesgo de atraso, la convergencia y la sensibilidad relevante. Se comparan tamaños solo si cambia la interpretación o la recomendación; el resto de tamaños, diagnósticos y experimentos va en el Anexo D.

## 4. Robustez, sensibilidad e insumos para S3

### 4.1 Observaciones influyentes (jackknife)

**Responsable:** Yerko · **Revisor:** José · **Desarrollo completo:** Anexo E

**Método.** Se conserva el análisis completo como referencia. Se aplica *jackknife* a todos los estimandos y coeficientes del inventario que admitan eliminar una fila: error estándar *jackknife* y cambio de cada estimación al retirar cada envío.

- **Medias:** se usa la identidad $\bar x_{(-i)} = (\text{suma} - x_i)/(n-1)$, que evita un cálculo cuadrático.
- **Efectos entre grupos:** se conserva la pertenencia al grupo y se actualizan los tamaños.
- **Pruebas:** se evalúa el cambio del efecto o del estadístico y de la decisión, sin interpretar el *jackknife* del valor *p* como un IC de un parámetro.
- **Casos indefinidos:** se registra la causa.

**Registro:** los diez registros más influyentes por estadístico, con `id_envio`, variable, valor original y cambio absoluto y relativo. Los empates se resuelven por ID y se informa que existen.

**Criterio.** Un valor extremo estadísticamente válido no se elimina por defecto.

**Interpretación** *(pendiente)*.

### 4.2 Sensibilidad a extremos y estructura

**Responsables:** Yerko y Sebastián · **Revisores:** José y Yerko · **Desarrollo completo:** Anexo E

**Escenarios**, evaluados por separado:

1. Datos completos (referencia).
2. Winsorización 1/99 de peso y costo.
3. Exclusión exploratoria de observaciones fuera de 1,5·IQR.
4. Análisis restringido a descuento ≤ 10.

Para cada escenario se registran la regla, los IDs afectados, el n retenido y los cambios en todos los estimandos, los IC, las siete pruebas, las 20 correlaciones y las 15 asociaciones. El CSV no se modifica, y los análisis restringidos no se tratan como estimaciones de la misma población.

**Recálculo por escenario.** Se repiten los IC *bootstrap* y las siete permutaciones en cada escenario válido, con B = 10.000, y se recalcula Holm para la familia completa del escenario. Si un escenario suprime un grupo (por ejemplo, E3 al restringir a descuento ≤ 10), se marca como «no evaluable por diseño», sin fabricar un contraste ni un resultado.

**Diagnósticos.** La comparación de la media con la mediana o la media recortada indica un cambio de estimando, no valida directamente el IC de la media.

**Segmento de descuento > 10.** En S1, los 2.647 envíos con descuento mayor que 10 están atrasados, frente a 3.916 de 8.352 con descuento ≤ 10 ($p = 0{,}468870$). Un *bootstrap* del primer segmento produce solo unos: IC [1, 1] y BCa posiblemente indefinido. No se interpreta como certeza poblacional; se conserva como advertencia el IC de Wilson de S1, [0,998551; 1]. El umbral no se cambia buscando el resultado más favorable.

**Para el cuerpo del informe** *(pendiente)*: qué conclusiones centrales se sostienen o cambian, con la magnitud y la evidencia de influencia, incluida toda discrepancia material que afecte la interpretación, aunque contradiga la conclusión inicial. Todos los cambios se publican en el Anexo E; no se trasladan al cuerpo todos los escenarios.

### 4.3 Sensibilidad de Monte Carlo

**Responsable:** José · **Revisor:** Sebastián · **Desarrollo completo:** Anexo D

**Sensibilidad a $p$.** Se varía $p$ en los extremos y el centro del IC de Wilson de S1, con $N = 1.000$ y $B \ge 10.000$ por escenario. Se miden los cambios en la media, el percentil 99 y $P(\text{tasa} > 0{,}60)$, y se presentan en un gráfico de sensibilidad. Esta actividad evidencia el indicador ID2.3.

**Otras entradas.** Para cada entrada adicional con IC en S1, se varían el centro y los extremos y se registra el efecto sobre su salida pertinente; si no hay IC en S1, se usan escenarios empíricos identificados como extensión. No se agregan parámetros ajustados fuera de S1.

**Contraste de estructura.** Se simula una composición fija de los dos segmentos de descuento, a partir de sus tamaños y tasas en S1:

- $N_{alto} = \text{redondeo}(N \cdot 2.647/10.999)$
- $K_{alto} \sim \text{Binomial}(N_{alto}, 1)$
- $K_{bajo} \sim \text{Binomial}(N - N_{alto},\ 3.916/8.352)$

Su dispersión se contrasta con la del modelo global. Se explicita que $p_{alto} = 1$ es una frontera muestral y una limitación.

**Interpretación** *(pendiente)*.

### 4.4 Reporte de resultados validados

**Responsable:** José · **Revisor:** Yerko · **Desarrollo completo:** Anexo F

**Archivos:** `outputs/resultados_validados.csv` (entrada a S3) y su síntesis legible, `docs/resultados_validados.md`.

**Campos por resultado:** ID, fuente y valor en S1, método en S2, n/B, estimación, IC o *p* en S2, diagnóstico de sensibilidad, estado, limitación, recomendación para S3, evidencia, responsable, revisor, commit y semilla.

**Estados permitidos:** *pendiente*, *respaldado bajo supuestos*, *sensible*, *no concluyente* y *no evaluable con justificación*. Se asignan solo después de ejecutar y revisar.

**Cobertura.** Se comprueba una cobertura del 100 % del inventario: cada resultado tiene una validación o una excepción metodológica explícita. El Anexo F contiene la matriz completa; el cuerpo del informe incluye únicamente los resultados y recomendaciones para S3 que derivan de la discusión central, con evidencia e incertidumbre.

**Orientación para S3:** variables a conservar, registros a inspeccionar, tratamientos a comparar y validación externa necesaria. No se declara causalidad, independencia ni transportabilidad a partir de la concordancia entre métodos.

## 5. Comprobaciones de aceptación

Antes del congelamiento final se verifica:

- [ ] Huellas y referencias de S1; los valores de referencia se reproducen con una tolerancia explícita de punto flotante.
- [ ] B ≥ 10.000 en cada técnica; dominios e IC válidos (coeficientes en [−1, 1] y V de Cramér en [0, 1]).
- [ ] Colas de *t*, *z*, *U* y $\chi^2$; Holm; emparejamiento del *bootstrap* y advertencias de BCa.
- [ ] Monte Carlo: MCSE, convergencia y controles binomiales.
- [ ] Revisión final del cuaderno completo por todo el equipo.
- [ ] Kernel reiniciado y todas las celdas ejecutadas en orden, repetido en un segundo equipo; cifras cotejadas con las tablas exportadas.
- [ ] Cobertura completa en los anexos y, en el cuerpo del informe, al menos tres parámetros, una prueba y entre tres y cinco correlaciones con evidencia y discusión, dentro de siete páginas.